##TUGAS
1. Buatlah model SVM dengan menggunakan data voice.csv dengan ketentuan,

    a. Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.

1.   Gunakan model dengan kernel linier.
2.   Gunakan model dengan kernel polynomial.
3.   Gunakan model dengan kernel RBF.

    b. Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.

2. Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!

###Klasifikasi voice.csv

In [81]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

data = pd.read_csv('voice.csv')

X = data.drop('label', axis=1)
y = data['label'].map({
    'male': 0,
    'female': 1
})

print("Shape:", data.shape)
print("\nJumlah label:")
print(y.value_counts())

Shape: (3168, 21)

Jumlah label:
label
0    1584
1    1584
Name: count, dtype: int64


###SVM rasio 70:30

In [82]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

models = {
    'Linear': Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='linear'))
    ]),

    'Polynomial': Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='poly'))
    ]),

    'RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf'))
    ])
}

hasil_70 = {}

for nama, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    hasil_70[nama] = accuracy_score(y_test, pred)

print("Hasil rasio 70:30:")
for kernel, akurasi in hasil_70.items():
    print(kernel, ":", round(akurasi * 100, 2), "%")

Hasil rasio 70:30:
Linear : 96.95 %
Polynomial : 94.95 %
RBF : 98.11 %


###SVM rasio 80:20

In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

hasil_80 = {}

for nama, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    hasil_80[nama] = accuracy_score(y_test, pred)

print("Hasil rasio 80:20:")
for kernel, akurasi in hasil_80.items():
    print(kernel, ":", round(akurasi * 100, 2), "%")

Hasil rasio 80:20:
Linear : 96.85 %
Polynomial : 95.27 %
RBF : 97.95 %


###Perbandingan hasil

In [84]:
hasil_svm = pd.DataFrame({
    'Kernel': ['Linear', 'Polynomial', 'RBF'],
    '70:30': [
        hasil_70['Linear'],
        hasil_70['Polynomial'],
        hasil_70['RBF']
    ],
    '80:20': [
        hasil_80['Linear'],
        hasil_80['Polynomial'],
        hasil_80['RBF']
    ]
})

hasil_svm['70:30'] = (hasil_svm['70:30'] * 100).round(2)
hasil_svm['80:20'] = (hasil_svm['80:20'] * 100).round(2)

hasil_svm

,Kernel,70:30,80:20
0,Linear,96.95,96.85
1,Polynomial,94.95,95.27
2,RBF,98.11,97.95


##2. Klasifikasi Citra Siang/Malam

###Load gambar dan fungsi histogram

In [91]:
import cv2
import numpy as np
from pathlib import Path

train_dir = "/content/images/images/training/"
test_dir = "/content/images/images/test/"

def load_images(img_dir):
    data = []
    path = Path(img_dir)

    for label_dir in path.iterdir():
        if label_dir.is_dir():
            label = label_dir.name

            for file in label_dir.glob("*.jpg"):
                img = cv2.imread(str(file))

                if img is not None:
                    data.append((img, label))

    return data


images = load_images(train_dir) + load_images(test_dir)


def histogram_feature(image):
    fitur = []

    # Histogram untuk channel B, G, dan R
    for channel in range(3):
        hist = cv2.calcHist(
            [image],
            [channel],
            None,
            [256],
            [0, 256]
        )

        hist = cv2.normalize(hist, hist).flatten()
        fitur.extend(hist)

    return fitur


print("Jumlah gambar:", len(images))

Jumlah gambar: 400


###Ekstraksi histogram dan label

In [92]:
label_encoder = {
    'night': 0,
    'day': 1
}

X_img = np.array([
    histogram_feature(img)
    for img, label in images
])

y_img = np.array([
    label_encoder[label]
    for img, label in images
])

print("Shape fitur :", X_img.shape)
print("Shape label :", y_img.shape)

print("\nJumlah kelas:")
print(pd.Series(y_img).value_counts())

Shape fitur : (400, 768)
Shape label : (400,)

Jumlah kelas:
1    200
0    200
Name: count, dtype: int64


###SVM RBF dengan rasio 80:20

In [95]:
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img,
    y_img,
    test_size=0.20,
    random_state=42,
    stratify=y_img
)

model_img = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])

model_img.fit(X_train_img, y_train_img)

pred_img = model_img.predict(X_test_img)

acc_img = accuracy_score(y_test_img, pred_img)

print("Akurasi SVM RBF:", round(acc_img * 100, 2), "%")

Akurasi SVM RBF: 100.0 %


###Hyperparameter tuning

In [96]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 0.001, 0.01, 0.1]
}

grid = GridSearchCV(
    model_img,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train_img, y_train_img)

best_pred = grid.best_estimator_.predict(X_test_img)
best_acc = accuracy_score(y_test_img, best_pred)

print("Parameter terbaik:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_ * 100, 2), "%")
print("Test accuracy setelah tuning:", round(best_acc * 100, 2), "%")

Parameter terbaik: {'svm__C': 10, 'svm__gamma': 'scale'}
Best CV accuracy: 99.38 %
Test accuracy setelah tuning: 100.0 %
